# Reproduce the Anthropochrony results on CPU

This notebook rebuilds the main table (T2) and the hero figure from the released `analysis_rows.jsonl`, using the same metric code as the pipeline. It needs no GPU, no model download and no credentials. Set `ANTHROPOCHRONY_DATA` to a folder holding `analysis_rows.jsonl`, `metrics.json`, `hypotheses.json` and `simulations.json`, or let it download them from the public dataset repo in `ANTHROPOCHRONY_DATASET`.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from anthropochrony.eval.metrics import summary, task_records
from anthropochrony.eval.stats import stratified_bootstrap

def locate():
    for candidate in (os.environ.get('ANTHROPOCHRONY_DATA'), 'results', '../results'):
        if candidate and (Path(candidate) / 'analysis_rows.jsonl').exists():
            return Path(candidate)
    from huggingface_hub import snapshot_download
    return Path(snapshot_download(os.environ['ANTHROPOCHRONY_DATASET'], repo_type='dataset',
                                  allow_patterns=['results/*'])) / 'results'

DATA = locate()
rows = [json.loads(line) for line in (DATA / 'analysis_rows.jsonl').read_text().splitlines()]
print(len(rows), 'estimate rows from', DATA)

## Main table: `test`, reference speed card, self frame

Errors are `|log10(estimate / measured)|`, averaged over question templates within a task. Fine-tuned models are scored against their own re-measured runs, the base model and baselines against the base runs. A parse failure counts as infinite error.

In [ ]:
def select(split='test', card='REF', frame='self'):
    groups = {}
    for row in rows:
        if row['split'] == split and row['card_id'] == card and row.get('frame', 'self') == frame:
            groups.setdefault((row['estimator'], row.get('target_source')), []).append(row)
    return groups

def table(split='test', resamples=2000):
    out = []
    for (estimator, source), group in sorted(select(split).items(), key=str):
        records = task_records(group)
        s = summary(records)
        lo, hi = stratified_bootstrap(
            records, lambda rs: float(np.median([r['abs_error'] for r in rs])), resamples=resamples)
        out.append({'estimator': estimator, 'targets': source or 'base', 'tasks': s['n_tasks'],
                    'MdAE log10': s['mdae_log10'], 'CI95': f'[{lo:.3f}, {hi:.3f}]',
                    'within 2x': s['within2x'], 'Spearman': s['spearman'],
                    'GM ratio': s['gm_ratio'], 'AUR': s['aur'], 'parse': s['parse_rate']})
    return pd.DataFrame(out)

t2 = table('test')
t2.round(3)

## Check against the released metrics

The released `metrics.json` was written with 10,000 resamples. The point estimates must match exactly; the confidence intervals match up to bootstrap noise.

In [ ]:
released = {(m['estimator'], m['split'], m['card_id'], m['frame'], m['target_source']): m
            for m in json.loads((DATA / 'metrics.json').read_text())}
bad = 0
for (estimator, source), group in select('test').items():
    mine = summary(task_records(group))['mdae_log10']
    ref = released[(estimator, 'test', 'REF', 'self', source or 'base')]['mdae_log10']
    bad += abs(mine - ref) > 1e-9
print('mismatching estimators:', bad)
assert bad == 0

## Estimated versus measured completion time (hero figure)

In [ ]:
chosen = [e for e in ('E1', 'M2-S0', 'M1-S0') if any(k[0] == e for k in select('test'))]
fig, axes = plt.subplots(1, len(chosen), figsize=(4.2 * len(chosen), 4), squeeze=False)
for axis, name in zip(axes.flat, chosen):
    group = [r for (e, _), g in select('test').items() if e == name for r in g]
    records = [r for r in task_records(group) if r['predicted']]
    x = np.array([r['actual'] for r in records]); y = np.array([r['predicted'] for r in records])
    axis.scatter(x, y, s=12, alpha=.55)
    line = np.geomspace(min(x.min(), y.min()) * .7, max(x.max(), y.max()) * 1.4, 50)
    axis.plot(line, line, 'k', lw=1); axis.fill_between(line, line / 2, line * 2, alpha=.15)
    axis.set(xscale='log', yscale='log', title=name, xlabel='measured (s)', ylabel='estimated (s)')
plt.tight_layout()

## Pre-registered hypotheses

In [ ]:
hypotheses = json.loads((DATA / 'hypotheses.json').read_text())
pd.DataFrame([{'hypothesis': k, 'status': v.get('status'), 'passed': v.get('passed')}
              for k, v in hypotheses.items()])

## Scheduling and timeout simulations (exploratory)

In [ ]:
path = DATA / 'simulations.json'
if path.exists():
    sim = json.loads(path.read_text())
    display(pd.DataFrame([{'estimator': k, 'SJF vs oracle': v['scheduling']['sjf_relative_to_oracle'],
                           'FIFO vs oracle': v['scheduling']['fifo_relative_to_oracle']}
                          for k, v in sim.items()]).round(3))